In [15]:
import sys
import pathlib
import pandas as pd
from kalpa import loaders

PROJECT = pathlib.Path.cwd().parent          
ROOT = PROJECT / "data"/ "raw"
SRC = PROJECT / "src"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

files = [p for p in ROOT.rglob("*") if p.is_file()]
print(len(files), "files")

344 files


In [16]:
pat = pd.read_csv(ROOT / "patients" / "patients.csv.gz", dtype="string")
print(pat.shape)
pat.head()

(309217, 6)


,patient_id,patient_name,phone,dob,city,signup_date
0,PAHM0030088,Rahul Fernandes,+91 85778 98441,1988-02-02,Ahmedabad,2023-01-14
1,PAHM0043743,Ananya D'Souza,+91 84824 41827,1992-11-13,Ahmedabad,2025-10-02
2,PAHM0011366,Rohan Mehta,+91 77327 81379,1944-07-05,Ahmedabad,2023-01-18
3,PAHM0009021,Vihaan Pillai,+91 76067 53849,1958-03-06,Ahmedabad,2022-08-06
4,PAHM0052680,Meera Desai,+91 81849 12697,2005-11-01,Ahmedabad,2026-05-19


### Trying to find duplicates

In [25]:
pat.isna().sum()

patient_id      0
patient_name    0
phone           0
dob             0
city            0
signup_date     0
phone_norm      0
name_norm       0
person_key      0
dtype: int64

#### Phone key normalised

In [17]:
import re

mask = pat["phone"].str.match(r"^\+91 \d{5} \d{5}$", na=False)

print(f"{mask.sum():,} match '+91 xxxxx xxxxx'")
print(f"{(~mask).sum():,} do NOT match")

pat.loc[~mask, "phone"].head(10)

303,153 match '+91 xxxxx xxxxx'
6,064 do NOT match


50     7859495787
103    8395317579
124    9064331252
230    7322581033
387    7415588003
394    7848181868
420    8681760357
458    9772231718
488    8598280841
540    9901633551
Name: phone, dtype: string

In [18]:
pat["phone_norm"] = loaders.normalise_phone(pat["phone"])

#### Normalise the Name 

In [ ]:
pat["name_norm"] = pat["patient_name"].str.upper().str.replace(r"\s+", " ", regex=True).str.strip()


### Checking duplicates in DOB

In [ ]:

def dob_shape(d):
    if pd.isna(d):
        return "null"
    if re.match(r"^\d{4}-\d{2}-\d{2}$", d):
        return "YYYY-MM-DD"
    if re.match(r"^\d{2}/\d{2}/\d{4}$", d):
        return "DD/MM/YYYY or MM/DD/YYYY"
    return "other"

pat["dob"].apply(dob_shape).value_counts()

dob
YYYY-MM-DD    309217
Name: count, dtype: int64

### Creating Person Key to drop duplicates

In [ ]:
pat["person_key"] = pat.groupby(["phone_norm", "name_norm", "dob"]).ngroup()
dup_mask = pat.duplicated(subset=["person_key"], keep="first")
print(f"{len(pat):,} rows, {pat['person_key'].nunique():,} distinct people, {int(dup_mask.sum()):,} duplicate rows")
duplicates = pat[pat.duplicated(subset=["person_key"], keep=False)].sort_values("person_key")

309,217 rows, 303,153 distinct people, 6,064 duplicate rows


In [10]:
duplicates

,patient_id,patient_name,phone,dob,city,signup_date,phone_norm,name_norm,person_key
64762,PBLR0062126,Saanvi Shah,+91 70002 42527,1978-02-05,Bengaluru,2026-02-05,7000242527,SAANVI SHAH,22
65449,PBLR0062126D,SAANVI SHAH,7000242527,1978-02-05,Bengaluru,2026-02-05,7000242527,SAANVI SHAH,22
285605,PPUN0030381D,FARHAN PILLAI,7000486083,1989-06-29,Pune,2025-05-10,7000486083,FARHAN PILLAI,54
308785,PPUN0030381,Farhan Pillai,+91 70004 86083,1989-06-29,Pune,2025-05-10,7000486083,FARHAN PILLAI,54
8663,PAHM0053725D,SAANVI MENON,7000884955,1943-04-19,Ahmedabad,2026-06-25,7000884955,SAANVI MENON,86
...,...,...,...,...,...,...,...,...,...
162240,PCHE0048349D,FARHAN SINGH,9996637333,1983-12-23,Chennai,2025-11-28,9996637333,FARHAN SINGH,302833
42839,PAHM0001639D,SIDDHARTH REDDY,9996937998,1964-12-03,Ahmedabad,2022-11-30,9996937998,SIDDHARTH REDDY,302866
10445,PAHM0001639,Siddharth Reddy,+91 99969 37998,1964-12-03,Ahmedabad,2022-11-30,9996937998,SIDDHARTH REDDY,302866
33062,PAHM0039897D,JOSEPH FERNANDES,9999895946,1950-02-25,Ahmedabad,2025-06-25,9999895946,JOSEPH FERNANDES,303150


In [13]:
pat_rejected = pat[dup_mask].copy()
patients_processed = pat[~dup_mask].copy()

print(len(pat), len(patients_processed), len(pat_rejected))

309217 303153 6064


In [14]:
patients_processed 

,patient_id,patient_name,phone,dob,city,signup_date,phone_norm,name_norm,person_key
0,PAHM0030088,Rahul Fernandes,+91 85778 98441,1988-02-02,Ahmedabad,2023-01-14,8577898441,RAHUL FERNANDES,159256
1,PAHM0043743,Ananya D'Souza,+91 84824 41827,1992-11-13,Ahmedabad,2025-10-02,8482441827,ANANYA D'SOUZA,149703
2,PAHM0011366,Rohan Mehta,+91 77327 81379,1944-07-05,Ahmedabad,2023-01-18,7732781379,ROHAN MEHTA,73823
3,PAHM0009021,Vihaan Pillai,+91 76067 53849,1958-03-06,Ahmedabad,2022-08-06,7606753849,VIHAAN PILLAI,61039
4,PAHM0052680,Meera Desai,+91 81849 12697,2005-11-01,Ahmedabad,2026-05-19,8184912697,MEERA DESAI,119527
...,...,...,...,...,...,...,...,...,...
309212,PPUN0034141,Ananya Shah,+91 72322 72862,1999-07-07,Pune,2025-09-09,7232272862,ANANYA SHAH,23332
309213,PPUN0013928,Siddharth Menon,+91 72309 39949,1967-03-30,Pune,2023-03-22,7230939949,SIDDHARTH MENON,23207
309214,PPUN0000920,Ritika Bose,+91 77049 50138,1969-09-16,Pune,2024-12-24,7704950138,RITIKA BOSE,70991
309215,PPUN0038468,Aditi Sharma,+91 83440 39856,2004-08-15,Pune,2026-02-28,8344039856,ADITI SHARMA,135523


In [ ]:
OUT = ROOT.parent / "processed"
OUT.mkdir(exist_ok=True)

patients_processed.to_csv(OUT / "patients_clean.csv", index=False)


import os
size_mb = os.path.getsize(OUT / "patients_clean.csv") / 1e6
print(f"saved: {OUT / 'patients_clean.csv'} | {size_mb:.0f} MB | expected rows: {len(patients_processed):,}")

saved: /Users/vasudevkamathm/Public/IITGN/Data_analysis/project/project/processed/patients_clean.csv | 31 MB | expected rows: 303,153


In [19]:
CLEAN = ROOT.parent / "data" / "clean"
CLEAN.mkdir(parents=True, exist_ok=True)

patients_processed.to_csv(CLEAN / "patients_clean.csv.gz", index=False, compression="gzip")

import os
size_mb = os.path.getsize(CLEAN / "patients_clean.csv.gz") / 1e6
print(f"saved: {CLEAN / 'patients_clean.csv.gz'} | {size_mb:.1f} MB | rows: {len(patients_processed):,}")

saved: /Users/vasudevkamathm/Public/IITGN/Data_analysis/project/project/data/clean/patients_clean.csv.gz | 9.6 MB | rows: 303,153
